# Validation Accuracy (Notebook wrapper)

This notebook is a thin wrapper around `modeling/validate_model_accuracy.py` so you can run validation evaluation from the notebook UI.

It writes metrics/plots/predictions to `modeling/results/<TARGET>/validation_eval_<EXPOSURE_SET>__<MODEL_FAMILY>/`.


In [ ]:
# ---- Fixed from here on out (per your constraint) ----
# Only Full_pesticides_raw + XGBoost, and no fitting/tuning on validation rows.

TARGETS = ["CASTHMA", "COPD"]
EXPOSURE_SET = "Full_pesticides_raw"
MODEL_FAMILY = "XGBoost (tuned)"
VALIDATION_SET = "external_holdout"  # train split only, evaluate on validation split

# CV is only used *inside the train split* to pick hyperparameters.
CV_SPLITS = 5
BOOTSTRAP_N = 100
RANDOM_STATE = 42

# Optional: where outputs should be written
OUTPUT_DIR = None  # defaults to modeling/results

# Website export (optional)
EXPORT_WEB_JSON = True
WEB_DISPLAY_YEAR = 2019

# Guardrails
assert EXPOSURE_SET == "Full_pesticides_raw"
assert MODEL_FAMILY == "XGBoost (tuned)"
assert VALIDATION_SET == "external_holdout"

print("Running:")
print("  TARGETS      =", TARGETS)
print("  EXPOSURE_SET =", EXPOSURE_SET)
print("  MODEL_FAMILY =", MODEL_FAMILY)
print("  VALIDATION   =", VALIDATION_SET)


In [ ]:
import subprocess
import sys
from pathlib import Path


REPO_ROOT = Path.cwd()
# If you run this notebook from within `modeling/`, step up one.
if (REPO_ROOT / "model_evaluation.py").exists():
    REPO_ROOT = REPO_ROOT.parent

script_path = REPO_ROOT / "modeling" / "validate_model_accuracy.py"
if not script_path.exists():
    raise FileNotFoundError(f"Could not find script: {script_path}")

OUT_BASE = (REPO_ROOT / "modeling" / "results") if OUTPUT_DIR is None else Path(OUTPUT_DIR)
model_family_safe = MODEL_FAMILY.replace(" ", "_")

out_dirs = {}
for target in TARGETS:
    cmd = [
        sys.executable,
        str(script_path),
        "--target",
        target,
        "--exposure-set",
        EXPOSURE_SET,
        "--model-family",
        MODEL_FAMILY,
        "--validation-set",
        VALIDATION_SET,
        "--cv-splits",
        str(CV_SPLITS),
        "--bootstrap-n",
        str(BOOTSTRAP_N),
        "--random-state",
        str(RANDOM_STATE),
        "--export-all-counties",
    ]
    if OUTPUT_DIR is not None:
        cmd += ["--output-dir", str(OUTPUT_DIR)]

    print("\nCommand:\n", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)

    out_dir = OUT_BASE / target / f"validation_eval_{EXPOSURE_SET}__{model_family_safe}"
    out_dirs[target] = out_dir

print("\nOutput dirs:")
for k, v in out_dirs.items():
    print(" ", k, "->", v)


In [ ]:
import pandas as pd

rows = []
for target, out_dir in out_dirs.items():
    metrics_path = out_dir / "metrics.csv"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Metrics file not found: {metrics_path}")
    rows.append(pd.read_csv(metrics_path))

metrics_all = pd.concat(rows, ignore_index=True)
metrics_all


In [ ]:
# Optional: show the first few prediction rows per target
from IPython.display import display
import pandas as pd

for target, out_dir in out_dirs.items():
    pred_files = sorted(list(out_dir.glob("predictions_*.csv")))
    print("\n", target)
    print("  Prediction files:", [p.name for p in pred_files])
    if pred_files:
        display(pd.read_csv(pred_files[0]).head())


In [ ]:
# Optional: export *all-county* estimates for the website map
# Produces web/data/validation_map_data.json with the same structure as web/data/xgboost_map_data.json.
#
# IMPORTANT: This does NOT fit on validation rows.
# It uses `predictions_all_counties.csv` written by the script after training on the train split.

import json
import pandas as pd


def min_max_norm(values: list[float]) -> list[float]:
    lo, hi = float(min(values)), float(max(values))
    if hi <= lo:
        return [0.5 for _ in values]
    return [(float(v) - lo) / (hi - lo) for v in values]


if not EXPORT_WEB_JSON:
    print("EXPORT_WEB_JSON=False; skipping.")
else:
    out = {}

    for target, out_dir in out_dirs.items():
        all_path = out_dir / "predictions_all_counties.csv"
        if not all_path.exists():
            raise FileNotFoundError(
                f"Missing {all_path}. Re-run the evaluation cell; it should call the script with --export-all-counties."
            )

        df = pd.read_csv(all_path)
        df = df[df["YEAR"].astype(int) == int(WEB_DISPLAY_YEAR)].copy()
        df["FIPS"] = df["FIPS"].astype(int).astype(str).str.zfill(5)

        key = target.lower() if target.lower() != "casthma" else "casthma"
        preds = df["prediction"].astype(float).tolist()
        risks = min_max_norm(preds)

        for fips, actual, pred, risk in zip(df["FIPS"], df["actual"], df["prediction"], risks):
            if fips not in out:
                out[fips] = {"year": int(WEB_DISPLAY_YEAR)}
            out[fips][key] = {
                "actual": float(actual),
                "prediction": float(pred),
                "risk_index": float(risk),
            }

    web_out = REPO_ROOT / "web" / "data" / "validation_map_data.json"
    web_out.parent.mkdir(parents=True, exist_ok=True)
    with open(web_out, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=0)

    print(f"Wrote {len(out)} counties to {web_out} (year {WEB_DISPLAY_YEAR})")
    print("Open the map with: web/map.html?source=validation")


## Additional evaluations (mirrors `model_evaluation.ipynb`)

The cells below load the saved validation predictions and run the same diagnostics used in `modeling/model_evaluation.ipynb`:
- Core plots (pred vs actual, residual diagnostics)
- Subgroup error tables (where metadata is available)
- Top-error inspection
- County-level mean absolute error maps (Plotly choropleth)


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Ensure repo imports work no matter where notebook runs from
# Walk upwards from CWD until we find the repo root containing `modeling/`.
if "REPO_ROOT" not in globals():
    here = Path.cwd().resolve()
    REPO_ROOT = None
    for p in [here, *here.parents]:
        if (p / "modeling" / "model_evaluation.py").exists():
            REPO_ROOT = p
            break
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Could not locate repo root (missing modeling/model_evaluation.py). "
            f"Started search from: {here}"
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modeling.model_evaluation import evaluate, detect_task  # noqa: E402


def _qsplit(series: pd.Series, *, q: float = 0.5, high_label: str = "high", low_label: str = "low", invert: bool = False) -> np.ndarray:
    s = pd.to_numeric(pd.Series(series), errors="coerce")
    thr = s.quantile(q)
    if invert:
        out = np.where(s <= thr, high_label, low_label)
    else:
        out = np.where(s >= thr, high_label, low_label)
    out[pd.isna(s)] = None
    return out


def _string_labels(s: pd.Series, missing_label: str = "missing") -> np.ndarray:
    out = pd.Series(s).astype("string").fillna(missing_label)
    return out.astype(str).to_numpy()


def load_validation_predictions(out_dir: Path) -> pd.DataFrame:
    p = out_dir / "predictions_validation_holdout.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    df = pd.read_csv(p)
    if "FIPS" in df.columns:
        df["FIPS"] = df["FIPS"].astype(int)
    if "YEAR" in df.columns:
        df["YEAR"] = df["YEAR"].astype(int)
    return df


def load_all_county_predictions(out_dir: Path) -> pd.DataFrame:
    p = out_dir / "predictions_all_counties.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    df = pd.read_csv(p)
    if "FIPS" in df.columns:
        df["FIPS"] = df["FIPS"].astype(int)
    if "YEAR" in df.columns:
        df["YEAR"] = df["YEAR"].astype(int)
    return df


def build_subgroup_columns_for_validation(pred_df: pd.DataFrame) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    """Mirror `model_evaluation.ipynb` subgroup definitions, but for validation rows."""
    data_path = Path(REPO_ROOT) / "data" / "validation.csv"
    if not data_path.exists():
        merged = pred_df.copy()
    else:
        val_df = pd.read_csv(data_path)
        merge_cols = [c for c in ["FIPS", "YEAR"] if c in pred_df.columns and c in val_df.columns]
        merged = pred_df.merge(val_df, on=merge_cols, how="left") if merge_cols else pred_df.copy()

    subgroup_columns: dict[str, np.ndarray] = {}

    # Poverty: prefer explicit poverty_rate; otherwise proxy with median_income.
    if "poverty_rate" in merged.columns:
        subgroup_columns["ej_poverty"] = _string_labels(
            _qsplit(merged["poverty_rate"], high_label="high_poverty", low_label="low_poverty")
        )
    elif "pct_poverty" in merged.columns:
        subgroup_columns["ej_poverty"] = _string_labels(
            _qsplit(merged["pct_poverty"], high_label="high_poverty", low_label="low_poverty")
        )
    elif "median_income" in merged.columns:
        subgroup_columns["ej_poverty"] = _string_labels(
            _qsplit(merged["median_income"], high_label="high_poverty", low_label="low_poverty", invert=True)
        )

    # SVI
    for svi_col in ["SVI", "svi", "svi_total", "svi_overall", "svi_rank"]:
        if svi_col in merged.columns:
            subgroup_columns["ej_svi"] = _string_labels(_qsplit(merged[svi_col], high_label="high_SVI", low_label="low_SVI"))
            break

    # Racial majority: person-centered labels (majority_white vs majority_BIPOC), no default to white.
    if "pct_white" in merged.columns:
        pw = pd.to_numeric(merged["pct_white"], errors="coerce")
        subgroup_columns["ej_racial_majority"] = _string_labels(
            np.where(pw >= 50, "majority_white", "majority_BIPOC")
        )

    # Rural/urban proxy via nchs_urban_rural.
    if "nchs_urban_rural" in merged.columns:
        ur = pd.to_numeric(merged["nchs_urban_rural"], errors="coerce")
        subgroup_columns["ej_rural"] = _string_labels(np.where(ur >= 5, "rural", "urban"))

    # Pesticide use proxy via pesticide_total_kg per cropland acre.
    if "pesticide_total_kg" in merged.columns and "cropland_acres" in merged.columns:
        denom = pd.to_numeric(merged["cropland_acres"], errors="coerce").replace(0, np.nan)
        num = pd.to_numeric(merged["pesticide_total_kg"], errors="coerce")
        per_acre = num / denom
        subgroup_columns["ej_pesticide_use"] = _string_labels(
            _qsplit(per_acre, high_label="high_pesticide_use", low_label="low_pesticide_use")
        )

    # IPM breadth
    if "ipm_breadth_acre" in merged.columns:
        subgroup_columns["ej_ipm_breadth"] = _string_labels(
            _qsplit(merged["ipm_breadth_acre"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
        )
    elif "ipm_breadth_value" in merged.columns:
        subgroup_columns["ej_ipm_breadth"] = _string_labels(
            _qsplit(merged["ipm_breadth_value"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
        )

    # Always include a simple place subgroup
    if "state_fips" in merged.columns:
        subgroup_columns["state_fips"] = merged["state_fips"].to_numpy()

    return subgroup_columns, merged


def run_notebook_evaluation(pred_df: pd.DataFrame) -> dict:
    subgroup_columns, merged = build_subgroup_columns_for_validation(pred_df)

    y_true = merged["actual"].to_numpy(dtype=float)
    y_pred = merged["prediction"].to_numpy(dtype=float)

    # Simple “high/low” splits as fallbacks
    subgroup_columns["actual_high"] = _string_labels(_qsplit(merged["actual"], q=0.5))
    subgroup_columns["pred_high"] = _string_labels(_qsplit(merged["prediction"], q=0.5))

    time_column = "YEAR" if "YEAR" in merged.columns else None

    return evaluate(
        y_true,
        y_pred,
        task=detect_task(y_true),
        X_test=merged,
        subgroup_columns=subgroup_columns if subgroup_columns else None,
        time_column=time_column,
        top_n_errors=25,
        min_subgroup_support=10,
    )


In [ ]:
# Run diagnostics per target (validation holdout)

for target, out_dir in out_dirs.items():
    print("\n" + "=" * 90)
    print(f"Target: {target}")
    print(f"Out dir: {out_dir}")

    pred_val = load_validation_predictions(Path(out_dir))
    print("Validation predictions:", pred_val.shape)
    display(pred_val.head())

    result = run_notebook_evaluation(pred_val)

    print("\nHeadline metrics")
    display(result["metrics"])

    print("\nCore diagnostic plots")
    # `evaluate` returns matplotlib Figure objects
    for fig, title in result.get("figures", []):
        fig.suptitle(f"{target} — {title}", y=1.02)
        plt.show(fig)

    if result.get("subgroup_tables"):
        print("\nSubgroup performance")
        for name, tbl in result["subgroup_tables"].items():
            print(f"\nSubgroup: {name}")
            display(tbl.sort_values("MAE", ascending=False).head(25))

    print("\nTop errors")
    display(result["top_errors"].head(25))

    print("\nInterpretation")
    print(result.get("interpretation", ""))
    print("\nNext steps")
    print(result.get("next_steps", ""))


In [ ]:
# County-level error choropleths (mirrors model_evaluation.ipynb section 4b)
# Uses `predictions_all_counties.csv` written by the validation script.

import plotly.express as px

COUNTIES_GEOJSON_URL = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

for target, out_dir in out_dirs.items():
    df = load_all_county_predictions(Path(out_dir)).copy()

    # Build per-county mean absolute error
    df["abs_residual"] = (df["actual"].astype(float) - df["prediction"].astype(float)).abs()
    county_error = (
        df.groupby(df["FIPS"].astype(str).str.zfill(5))
        .agg(mean_abs_error=("abs_residual", "mean"), n=("abs_residual", "count"))
        .reset_index()
        .rename(columns={"FIPS": "fips"})
    )

    # Worst/best sets (top/bottom 20% by mean error)
    q_high = county_error["mean_abs_error"].quantile(0.80)
    q_low = county_error["mean_abs_error"].quantile(0.20)
    county_error["error_bucket"] = np.where(
        county_error["mean_abs_error"] >= q_high,
        "worst_20pct",
        np.where(county_error["mean_abs_error"] <= q_low, "best_20pct", "middle"),
    )

    print("\n" + "=" * 90)
    print(f"County error maps — {target}")

    fig1 = px.choropleth(
        county_error,
        geojson=COUNTIES_GEOJSON_URL,
        locations="fips",
        color="mean_abs_error",
        color_continuous_scale="Reds",
        scope="usa",
        hover_data={"fips": True, "mean_abs_error": ":.3f", "n": True},
        title=f"Mean absolute prediction error by county ({target})",
    )
    fig1.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0})
    fig1.show()

    fig2 = px.choropleth(
        county_error,
        geojson=COUNTIES_GEOJSON_URL,
        locations="fips",
        color="error_bucket",
        color_discrete_map={"worst_20pct": "#d7301f", "best_20pct": "#1a9850", "middle": "#cccccc"},
        scope="usa",
        hover_data={"fips": True, "mean_abs_error": ":.3f", "n": True},
        title=f"Worst / best prediction counties (top & bottom 20% by mean error) — {target}",
    )
    fig2.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0})
    fig2.show()

    print("\nWorst counties (by mean_abs_error)")
    display(county_error.sort_values("mean_abs_error", ascending=False).head(15))
    print("\nBest counties (by mean_abs_error)")
    display(county_error.sort_values("mean_abs_error", ascending=True).head(15))
